# Week 18: Retrieval-Augmented Generation (RAG) - Part 2

## Measuring and Improving the Week 17 Pipeline

## Learning Objectives

By the end of this session, you will be able to:
1. **Compare chunking strategies** (fixed, recursive, hierarchical) and measure their impact on retrieval quality
2. **Add a Bedrock reranker** (Cohere Rerank 3.5) on top of a Strands retriever and quantify the precision lift
3. **Evaluate RAG outputs with RAGAS v0.4** (faithfulness, answer_relevancy, context_precision) using Bedrock Claude Haiku as the judge LLM - no external API keys
4. **Run an A/B comparison** of two RAG configurations end-to-end, pick a winner with data, and roll it back into the Week 17 supervisor

## Prerequisites

- Completed Week 17 (agentic RAG, Strands `policy_retriever_agent`, `mem0_memory`, Bedrock KB via `strands_tools.retrieve`)
- Comfortable with `strands_tools.retrieve`, Bedrock Converse API, SageMaker session + `get_execution_role()`
- Watched pre-class videos on chunking, reranking, RAGAS
- Have `STRANDS_KNOWLEDGE_BASE_ID` set (from Week 17 - same class KB)

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & Week 17 Recap | 10 min | Code |
| Section 1: Chunking Strategies | 25 min | Demo + Lab 1 |
| Section 2: Reranking with Cohere Rerank 3.5 | 20 min | Demo + Lab 2 |
| Section 3: RAG Evaluation with RAGAS | 25 min | Demo + Lab 3 |
| Section 4: A/B Pipeline Optimization (MAIN OUTCOME) | 25 min | Demo + Main Lab 4 |
| Wrap-up & Homework | 5 min | Markdown |

## The Story So Far

In Week 17 you built a multi-retriever fraud supervisor. It WORKS, but how do you know it works WELL? Your manager asks: "Is our RAG pipeline reliable enough to replace the human-reviewed policy lookups?" You cannot answer with a demo. You need measurement.

This week you pull the three levers that move RAG quality in production:

```mermaid
graph LR
    subgraph "Week 17: WORKS"
        W17[policy_retriever_agent<br/>default chunking<br/>no reranking<br/>no evaluation]
    end

    subgraph "Week 18: WORKS WELL"
        C[Chunking<br/>fixed vs recursive<br/>vs hierarchical]
        R[Reranking<br/>Cohere Rerank 3.5]
        E[Evaluation<br/>RAGAS faithfulness<br/>answer_relevancy<br/>context_precision]
        AB[A/B Comparison<br/>pick a winner<br/>with numbers]
    end

    W17 --> C
    W17 --> R
    C --> AB
    R --> AB
    E --> AB
    AB --> OUT[Optimized supervisor<br/>defended with metrics]

    style OUT fill:#e8f5e9,stroke:#4caf50,stroke-width:3px
```

The take-home is not a new agent. It is a TUNED version of Week 17's supervisor, plus the measurement discipline to defend it in a review.

## This Week vs Next Week

| | **Week 18 (Today)** | **Week 19 (Next Week)** |
|---|---|---|
| **Focus** | Measure and improve ONE RAG pipeline | Version and track MANY experiments |
| **You build** | A/B comparison DataFrame, pick winner | DVC-versioned data, MLflow-tracked runs |
| **Key concepts** | Chunking, reranking, RAGAS metrics | Data versioning, experiment tracking |
| **Why it matters** | You can defend your config with numbers | You can reproduce last month's config |

## GPU Setup

No GPU needed. All work is API-based through Amazon Bedrock. A CPU SageMaker Studio notebook is sufficient.

# Section 0: Environment Setup & Week 17 Recap

We reuse the Week 17 environment (Strands, Bedrock, SageMaker role) and add three new libraries:

- `langchain` + `langchain-aws` - for local chunking experiments and as the Bedrock bridge RAGAS needs
- `langchain-community` - for `BedrockEmbeddings`
- `ragas==0.4.3` - the evaluation framework, with Bedrock Claude Haiku as the judge

Nothing new for reranking - the Bedrock Rerank API is already reachable through `boto3.client('bedrock-agent-runtime').rerank(...)`.

AWS credentials come from your SageMaker execution role - same pattern as Week 17. No `getpass`, no API keys to paste.

The setup cells below verify every piece (model access, KB id, rerank model) and fail loud if anything is missing.

In [ ]:
# Cohere Rerank 3.5 uses the bedrock-agent-runtime.rerank() API.
# No extra package needed - it is in boto3>=1.35.

# Install required libraries. Versions are pinned for reproducibility.
# sagemaker pinned to v2.257.3: v3 removed get_execution_role() from the top-level namespace.

%pip install -q \
    "sagemaker==2.257.3" \
    "strands-agents>=1.37" \
    "strands-agents-tools[mem0-memory]>=0.2.10" \
    "boto3>=1.35" \
    "langchain>=0.3" \
    "langchain-aws>=0.2" \
    "langchain-community>=0.3" \
    "ragas>=0.4" \
    "datasets>=2.18" \
    "faiss-cpu>=1.9,<2" \
    "rank_bm25>=0.2.2" \
    "numpy>=1.25,<3"

print("\nPackages installed. If this was your first install, RESTART THE KERNEL before running the next cell.")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
# IMPORTANT: strands_tools.retrieve reads env vars AT IMPORT TIME. We set the
# required ones (KNOWLEDGE_BASE_ID, MIN_SCORE, RETRIEVE_ENABLE_METADATA_DEFAULT)
# BEFORE the strands import so the tool works without a kernel restart.

import os
import json
import time
import boto3
import sagemaker
import pandas as pd
from sagemaker import get_execution_role
from importlib.metadata import version as pkg_version


# =============================================================================
# SET strands_tools.retrieve ENVIRONMENT VARIABLES (before import)
# =============================================================================
os.environ["KNOWLEDGE_BASE_ID"] = os.environ.get("KNOWLEDGE_BASE_ID") \
    or os.environ.get("STRANDS_KNOWLEDGE_BASE_ID") \
    or "FARSQGTONR"  # same class KB as Week 17
os.environ["STRANDS_KNOWLEDGE_BASE_ID"] = os.environ["KNOWLEDGE_BASE_ID"]
os.environ["MIN_SCORE"] = os.environ.get("MIN_SCORE", "0.2")
os.environ["RETRIEVE_ENABLE_METADATA_DEFAULT"] = "true"


# =============================================================================
# STRANDS + LANGCHAIN IMPORTS
# =============================================================================
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import retrieve

# Chunking + local indexing (NEW in Week 18)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_aws import BedrockEmbeddings, ChatBedrockConverse
from langchain_core.documents import Document

# RAGAS (NEW in Week 18)
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Version check
for pkg in ["strands-agents", "boto3", "langchain", "langchain-aws", "ragas", "faiss-cpu", "numpy"]:
    try:
        print(f"  {pkg:22s} {pkg_version(pkg)}")
    except Exception:
        print(f"  {pkg:22s} (not installed)")


# =============================================================================
# SAGEMAKER SESSION + EXECUTION ROLE (same pattern as Week 15/16/17)
# =============================================================================
sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

print(f"\nSageMaker execution role: {role.split('/')[-1]}")
print(f"AWS Region:               {AWS_REGION}")


# =============================================================================
# MODEL CONFIGURATION (same as Week 15/16/17)
# =============================================================================
MODEL_ID       = "us.anthropic.claude-3-haiku-20240307-v1:0"
EMBED_MODEL_ID = "amazon.titan-embed-text-v2:0"
RERANK_MODEL_ID  = "cohere.rerank-v3-5:0"
RERANK_MODEL_ARN = f"arn:aws:bedrock:{AWS_REGION}::foundation-model/{RERANK_MODEL_ID}"

llm = BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION)

langchain_llm      = ChatBedrockConverse(model=MODEL_ID, region_name=AWS_REGION)
bedrock_embeddings = BedrockEmbeddings(model_id=EMBED_MODEL_ID, region_name=AWS_REGION)

print(f"\nLLM:        {MODEL_ID}")
print(f"Embeddings: {EMBED_MODEL_ID}")
print(f"Reranker:   {RERANK_MODEL_ID} (Cohere Rerank 3.5 via Bedrock Rerank API)")


# =============================================================================
# BOTO3 CLIENTS
# =============================================================================
bedrock_runtime       = boto3.client("bedrock-runtime",       region_name=AWS_REGION)
bedrock_agent         = boto3.client("bedrock-agent",         region_name=AWS_REGION)
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)


# =============================================================================
# PRE-FLIGHT PROBES
# =============================================================================
# Probe 1: LLM access
try:
    probe = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print(f"\nLLM probe OK:    {probe['output']['message']['content'][0]['text']!r}")
except Exception as e:
    print(f"\nLLM probe FAILED: {e}")
    print(f"Ask your instructor to enable Bedrock access for {MODEL_ID}.")
    raise


# Probe 2: Week 17 shared Knowledge Base id
STRANDS_KNOWLEDGE_BASE_ID = os.environ["KNOWLEDGE_BASE_ID"]

try:
    kb_info = bedrock_agent.get_knowledge_base(knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID)
    print(f"KB probe OK:     {STRANDS_KNOWLEDGE_BASE_ID} ({kb_info['knowledgeBase']['name']})")
except Exception as e:
    print(f"KB probe FAILED: {e}")
    print("Ask your instructor for the correct Knowledge Base id.")
    raise


# Probe 3: Cohere Rerank 3.5 access
try:
    _rr = bedrock_agent_runtime.rerank(
        queries=[{"type": "TEXT", "textQuery": {"text": "CTR threshold"}}],
        sources=[{"type": "INLINE",
                  "inlineDocumentSource": {"type": "TEXT",
                      "textDocument": {"text": "A bank must file a CTR for cash over $10,000."}}}],
        rerankingConfiguration={
            "type": "BEDROCK_RERANKING_MODEL",
            "bedrockRerankingConfiguration": {
                "numberOfResults": 1,
                "modelConfiguration": {"modelArn": RERANK_MODEL_ARN},
            },
        },
    )
    print(f"Rerank probe OK: score={_rr['results'][0]['relevanceScore']:.3f}")
except Exception as _e:
    print(f"Rerank probe FAILED: {_e}")
    print(f"Ask your instructor to enable bedrock:Rerank for {RERANK_MODEL_ID}.")
    raise

print("\nEnvironment ready.")

In [ ]:
# =============================================================================
# WEEK 17 RECAP - RE-DECLARE AGENTS SO THIS NOTEBOOK STANDS ALONE
# =============================================================================
# We re-declare the three Week 17 agents here the same way Week 17 re-declared
# Week 15's TRANSACTION_DATABASE. This gives us the BASELINE to improve.
#
#   baseline_policy_retriever   -> becomes policy_retriever_v2 in Section 2
#   baseline_case_agent         -> becomes case_history_v2 in Section 3
#   lab3_supervisor             -> becomes week18_supervisor in Lab 4 Part B

import logging
logging.getLogger("mem0").setLevel(logging.CRITICAL)

baseline_policy_retriever = Agent(
    model=llm,
    tools=[retrieve],
    system_prompt=(
        "You are a fraud policy specialist. Always call retrieve before "
        "answering. Answer only from retrieved content. If the KB does not "
        "cover the question, say so and stop."
    ),
    callback_handler=None,
)

baseline_case_agent = Agent(
    model=llm,
    tools=[],  # mem0_memory imported below after env vars are set
    system_prompt=(
        "You are a case history specialist. Store and retrieve prior fraud "
        "investigation findings. Always scope by investigator_id."
    ),
    callback_handler=None,
)


# =============================================================================
# MEM0 MEMORY CONFIGURATION (for Lab 4 Part B multi-agent outcome)
# =============================================================================
# mem0_memory stores per-investigator case history. FAISS backend writes to
# /tmp - ephemeral on SageMaker Studio, which is fine for class sessions.
os.environ["MEM0_LLM_PROVIDER"]   = "aws_bedrock"
os.environ["MEM0_LLM_MODEL"]      = MODEL_ID
os.environ["MEM0_EMBED_PROVIDER"] = "aws_bedrock"
os.environ["MEM0_EMBED_MODEL"]    = EMBED_MODEL_ID

from strands_tools import mem0_memory

# Rebuild baseline_case_agent now that mem0_memory is imported
baseline_case_agent = Agent(
    model=llm,
    tools=[mem0_memory],
    system_prompt=(
        "You are a case history specialist. Store and retrieve prior fraud "
        "investigation findings. Always scope by investigator_id."
    ),
    callback_handler=None,
)

# =============================================================================
# LOCAL FRAUD CORPUS - same content as the Week 17 KB, as in-memory text
# so we can freely experiment with chunking. In production you would read
# this from S3; here we inline it for classroom use.
# =============================================================================
FRAUD_POLICY_CORPUS = [
    ("ctr_rules.md",
     "Currency Transaction Report (CTR) rules under 31 CFR 1010.311. Banks "
     "must file a CTR for each transaction in currency of more than $10,000. "
     "Multiple transactions are aggregated when known to be conducted by or "
     "on behalf of the same person and result in cash in or cash out totaling "
     "more than $10,000 in any one business day. Structuring - breaking a "
     "transaction into smaller amounts to evade the CTR threshold - is itself "
     "a federal violation under 31 USC 5324."),
    ("ofac_screening.md",
     "OFAC screening requirements. All wire transfers must be screened against "
     "the OFAC Specially Designated Nationals (SDN) list before execution. "
     "International wire transfers involving countries on the OFAC sanctions "
     "list (including but not limited to Iran, North Korea, Syria, and Cuba) "
     "require additional review and may be blocked outright. False positives "
     "must be cleared within 24 hours."),
    ("wire_record_keeping.md",
     "Wire transfer recordkeeping under 31 CFR 1010.410. For any international "
     "wire transfer of $3,000 or more, the bank must retain the originator's "
     "name, address, account number, amount, execution date, payment "
     "instructions, beneficiary bank, and beneficiary name. Records must be "
     "retained for five years."),
    ("structuring_red_flags.md",
     "Structuring red flags. Multiple cash deposits of amounts just under "
     "$10,000 across consecutive days at the same or related accounts are a "
     "classic structuring pattern. Velocity anomalies - for example three or "
     "more transactions at unrelated merchants within 30 minutes - indicate "
     "potential card testing or account takeover."),
    ("account_takeover_patterns.md",
     "Account takeover (ATO) indicators. A password change followed within "
     "minutes by a wire transfer to a newly added payee from an unfamiliar IP "
     "is a high-confidence ATO signal. Transactions that originate from "
     "geographies inconsistent with the customer's historical footprint, "
     "especially from countries the customer has never transacted with, "
     "require hold and verification."),
    ("unusual_hours_rule.md",
     "Unusual hours rule. Transactions initiated between 1:00 AM and 5:00 AM "
     "local time that fall outside the customer's typical active window "
     "require enhanced monitoring. Two or more such transactions within a "
     "single session should trigger a SAR review."),
    ("new_payee_hold.md",
     "New payee large transfer hold. Wire transfers exceeding $5,000 to payees "
     "that were added to the account within the preceding 72 hours require "
     "two-factor customer verification and a 24-hour hold regardless of the "
     "customer's risk score."),
    ("high_risk_merchant_categories.md",
     "High-risk merchant category codes (MCCs). Cryptocurrency exchanges, "
     "offshore gambling platforms, and money transfer services are classified "
     "as high-risk MCCs. Transactions at these merchants for amounts over "
     "$1,000 require enhanced due diligence."),
]

print("baseline_policy_retriever ready.")
print("baseline_case_agent ready.")
print(f"Local fraud corpus loaded: {len(FRAUD_POLICY_CORPUS)} policy documents.")
print("mem0_memory tool configured for Lab 4 Part B.")

# Section 1: Chunking - Step 1 of 3 Improvements

## Why Chunking Matters for the PolicyRetriever

Week 17's `policy_retriever_agent` uses the Bedrock KB with its DEFAULT chunking
(fixed-size, ~300 tokens). This week we measure whether different chunk sizes
change what the agent retrieves. In Section 2 we add Cohere Rerank 3.5 on top.
In Lab 2 we wire both improvements into `policy_retriever_v2`. In Lab 4 Part B
we assemble `week18_supervisor` - the upgraded version of Week 17's `lab3_supervisor`.

## Why Chunking Is a Lever

A chunk is the unit a retriever fetches. If the chunk is too small, the LLM loses context. If the chunk is too large, the retriever loses precision and costs rise. The 2026 published benchmark (Vecta, 50 academic papers) puts recursive character splitting at 512 tokens with 50-100 overlap at 69% end-to-end accuracy, outperforming more exotic methods. But NVIDIA's FinanceBench benchmark found 1024-token chunks outperformed 512 on financial documents - because tables and numerical context need to stay together.

Fraud policies look a lot like financial prose. Short general answer: start with recursive-512-80, but measure.

```mermaid
graph TB
    subgraph "Fixed"
        F1[300 tokens<br/>10% overlap] --> FX[Many small chunks<br/>loses cross-section context]
    end

    subgraph "Recursive"
        R1[512 tokens<br/>80 token overlap<br/>split on paragraph -> line -> space] --> RX[Benchmark default<br/>respects structure]
    end

    subgraph "Hierarchical"
        H1[Parent 1500 + Child 300<br/>retrieve child, return parent] --> HX[Production AWS default<br/>best retrieval + best context]
    end

    style FX fill:#ffebee,stroke:#c62828,color:#000
    style RX fill:#e8f5e9,stroke:#2e7d32,color:#000
    style HX fill:#e8f5e9,stroke:#2e7d32,color:#000
```

## Bedrock Native Chunking Options

Bedrock KBs accept a `ChunkingConfiguration` at ingestion time:

- `NONE` - treat each file as one chunk
- `FIXED_SIZE` - pick tokens + overlap percentage
- `HIERARCHICAL` - parent + child token budgets
- `SEMANTIC` - split on natural-language similarity boundaries (extra cost - uses an FM)

For this section we experiment LOCALLY with langchain splitters because we cannot re-ingest the shared class KB mid-lesson. After Lab 1 you will kick off a cell that ingests the fraud corpus into three separate Bedrock KBs (one per chunk config) so you can query real Bedrock infrastructure after the break.

In [ ]:
# =============================================================================
# DEMO: Split the Fraud Corpus Three Ways and Index Locally
# =============================================================================
# We build three local FAISS indexes over the SAME text so we can isolate
# the effect of chunking alone. The embedding model is Titan V2 in all
# three indexes. The only thing that changes is HOW we chop the text.

# Materialize the corpus as langchain Documents
documents = [Document(page_content=text, metadata={"source": name})
             for name, text in FRAUD_POLICY_CORPUS]

# ---- Config A: small fixed chunks ----
splitter_a = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks_a   = splitter_a.split_documents(documents)

# ---- Config B: benchmark default (recursive 512/80) ----
splitter_b = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=80)
chunks_b   = splitter_b.split_documents(documents)

# ---- Config C: fraud-domain tuned (recursive 1024/150) ----
splitter_c = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=150)
chunks_c   = splitter_c.split_documents(documents)

print(f"Config A (300/30):   {len(chunks_a)} chunks")
print(f"Config B (512/80):   {len(chunks_b)} chunks")
print(f"Config C (1024/150): {len(chunks_c)} chunks")

# Index each with Titan V2 embeddings into a local FAISS store
index_a = FAISS.from_documents(chunks_a, bedrock_embeddings)
index_b = FAISS.from_documents(chunks_b, bedrock_embeddings)
index_c = FAISS.from_documents(chunks_c, bedrock_embeddings)

print("\nThree FAISS indexes built with Titan V2 embeddings.")

In [ ]:
# =============================================================================
# DEMO: Query Each Chunk Config with Two Test Queries
# =============================================================================
# Test queries chosen to surface a difference:
#   Q1 is a direct lookup ("CTR threshold")       - small chunks should win
#   Q2 needs cross-section context ("password change then wire transfer")
#                                                 - larger chunks should win
#                                                   because they keep the
#                                                   sequence together

queries = [
    ("Q1 direct lookup",
     "What is the CTR reporting threshold?"),
    ("Q2 multi-context",
     "What indicates account takeover when a password change is followed by a wire transfer?"),
]

for label, q in queries:
    print(f"\n=== {label}: {q!r} ===")
    for name, idx in [("A (300/30)",   index_a),
                      ("B (512/80)",   index_b),
                      ("C (1024/150)", index_c)]:
        hits = idx.similarity_search_with_score(q, k=1)
        doc, score = hits[0]
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"  {name:15s} score={score:6.3f}  source={doc.metadata['source']}")
        print(f"                  text: {preview}...")

print("\nObservation: config choice depends on query shape.")
print("The lab below makes that observation measurable.")

## Lab 1: Chunking Trade-off Table (15 min)

### Your Task

Build a pandas DataFrame comparing how the THREE chunk configs from the demo perform across FOUR fraud test queries. This is the first time you produce metrics in this week's notebook.

### Steps

1. Define a list of 4 test queries covering different shapes (direct lookup, cross-section, multi-document, and an adversarial one the KB likely cannot answer).
2. For each (query, config) pair, retrieve the top-1 hit. Record:
   - config name
   - query label
   - retrieved source filename
   - similarity score
3. Assemble a pandas DataFrame with columns: `query`, `config`, `source`, `score`.
4. Print the DataFrame sorted by query so you can eyeball which config wins per query shape.

### Expected Output

A 12-row DataFrame (4 queries x 3 configs) with readable scores. You should see config C win on queries needing more context, config A win on direct lookups. Not every query has a single correct config.

### Stretch (for fast finishers)

Add a 4th config using `MarkdownHeaderTextSplitter` on headings. The fraud policy docs in the demo corpus use file-based separation that could be exploited by a header-aware splitter. Measure whether it wins on any query and explain why.

### Homework Extension

Replicate ONE of your local configs in Bedrock itself by creating a small second KB with `ChunkingConfiguration=HIERARCHICAL` (parent 1500, child 300, overlap 60). Ingest the same 8 policy docs. Compare retrieval quality against your local results. Why might Bedrock's hierarchical retrieval score DIFFERENTLY than local recursive splitting even though the words are identical?

In [ ]:
# =============================================================================
# SOLUTION: Lab 1 - Chunking Trade-off Table
# =============================================================================
# Four queries covering different shapes: direct lookup (small chunks win),
# cross-section (large chunks win), multi-policy, and adversarial (not in KB).
# The adversarial query tests whether the model hallucinates when the KB lacks
# the answer - an important compliance check.

lab1_queries = [
    ("direct_lookup",  "What is the CTR reporting threshold?"),
    ("cross_section",  "What indicates account takeover after a password change?"),
    ("multi_policy",   "What are the wire transfer recordkeeping requirements?"),
    ("adversarial",    "What is the $50,000 suspicious activity threshold?"),
]

rows = []
for label, q in lab1_queries:
    for cfg_name, idx in [("A (300/30)",   index_a),
                          ("B (512/80)",   index_b),
                          ("C (1024/150)", index_c)]:
        hits = idx.similarity_search_with_score(q, k=1)
        doc, score = hits[0]
        rows.append({
            "query":  label,
            "config": cfg_name,
            "source": doc.metadata["source"],
            "score":  round(score, 4),
        })

lab1_df = pd.DataFrame(rows).sort_values("query")
print(lab1_df.to_string(index=False))

# Expected observation:
# - direct_lookup: config A (300/30) wins - small chunks isolate the $10,000 sentence
# - cross_section: config C (1024/150) wins - keeps the password-change + wire sequence together
# - adversarial: all configs return something (no "not found") - this is why faithfulness
#   evaluation is critical; the model may hallucinate an answer from a nearby chunk

In [ ]:
# =============================================================================
# KICK OFF DURING BREAK - Three Chunking KBs (Instructor Pre-Provisioned)
# =============================================================================
# Your instructor created three Bedrock Knowledge Bases before class - one per
# chunk config from Lab 1. Ingestion is already complete. Run this cell to
# load the KB IDs so the next cell can query them.
#
# You do NOT need to create or ingest anything. Just run this cell.
# The three KBs use the same S3 Vectors bucket as the Week 17 class KB.

import time as _time

_ba = boto3.client("bedrock-agent", region_name=AWS_REGION)

chunking_kb_ids = {
    "a300":  {"kb_id": "GDAG1GTGE0", "ds_id": "TS0WRALMVI", "job_id": "KDFL1CBXHZ"},
    "b512":  {"kb_id": "0ZOUD8TMJO", "ds_id": "D818WX25GB", "job_id": "YZPWO4RO5J"},
    "c1024": {"kb_id": "AGRVXDTU7J", "ds_id": "RI5FWTCFNR", "job_id": "GRACIZRTWY"},
}

print("Chunking KB IDs loaded (instructor pre-provisioned):")
for name, info in chunking_kb_ids.items():
    print(f"  {name:6s}  kb_id={info['kb_id']}")
print("\nAll three KBs are COMPLETE and ready to query.")
print("Run the next cell to verify ingestion status and query results.")

In [ ]:
# =============================================================================
# AFTER BREAK - Poll Ingestion + Query the Three Chunk-Config KBs
# =============================================================================
# Run this cell after you come back from the break.
# It checks whether ingestion finished, then queries all three KBs.

def _wait_for_ingestion(kb_id: str, ds_id: str, job_id: str,
                        timeout_s: int = 300) -> str:
    """Poll until the ingestion job reaches a terminal state."""
    deadline = _time.time() + timeout_s
    while _time.time() < deadline:
        resp = _ba.get_ingestion_job(
            knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
        )
        status = resp["ingestionJob"]["status"]
        if status in ("COMPLETE", "FAILED", "STOPPED"):
            return status
        _time.sleep(15)
    return "TIMEOUT"


print("Checking ingestion status...")
for suffix, info in chunking_kb_ids.items():
    status = _wait_for_ingestion(info["kb_id"], info["ds_id"], info["job_id"])
    print(f"  {suffix:20s}  kb_id={info['kb_id']}  status={status}")


# Query all three KBs with the same two test queries from the demo.
_test_queries = [
    ("direct_lookup",  "What is the CTR reporting threshold for cash transactions?"),
    ("multi_context",  "What indicates account takeover when a password change is "
                       "followed by a wire transfer to a new payee?"),
]

print("\n--- Bedrock KB retrieval results per chunk config ---")
for qlabel, qtext in _test_queries:
    print(f"\n=== {qlabel}: {qtext[:80]!r} ===")
    for suffix, info in chunking_kb_ids.items():
        resp = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=info["kb_id"],
            retrievalQuery={"text": qtext},
            retrievalConfiguration={
                "vectorSearchConfiguration": {"numberOfResults": 1}
            },
        )
        hits = resp.get("retrievalResults", [])
        if hits:
            score = hits[0].get("score", 0.0)
            text  = hits[0].get("content", {}).get("text", "")[:120].replace("\n", " ")
            print(f"  {suffix:20s}  score={score:.3f}  text: {text}...")
        else:
            print(f"  {suffix:20s}  no results")

print("\nObservation: compare these Bedrock scores to your local FAISS scores from Lab 1.")
print("Do the rankings match? If not, why might Bedrock chunk differently than langchain?")

# Clean up - delete the three temporary KBs and their S3 Vectors indexes.
# The Week 17 KB (FARSQGTONR) is NOT touched.
print("\nCleaning up temporary KBs...")
for suffix, info in chunking_kb_ids.items():
    try:
        _ba.delete_knowledge_base(knowledgeBaseId=info["kb_id"])
        print(f"  Deleted KB: {info['kb_id']} ({suffix})")
    except Exception as _e:
        print(f"  Could not delete {info['kb_id']}: {_e}")
print("Done. The main class KB (FARSQGTONR) is untouched.")

> **Think About It #1**: Your Lab 1 table shows config C wins on multi-context queries and config A wins on direct lookups. In production you cannot ship three retrievers - you ship one. Do you pick the config that wins on the most QUERIES, or the config whose LOSSES are least damaging for compliance? (Hint: in a regulated domain, a missed policy reference is worse than an extra chunk of irrelevant text.) How would you decide, and what would you write in the PR description to justify it?

# Section 2: Reranking - Step 2 of 3 Improvements

## Adding Cohere Rerank 3.5 to the PolicyRetriever

The demo in this section replaces the bi-encoder's top-k ordering with the
actual Bedrock Rerank API (Cohere Rerank 3.5). After Lab 2 you will have
`policy_retriever_v2` - a Strands Agent that calls a `@tool` wrapping
`reranked_retrieve`. In Lab 4 Part B we assemble `week18_supervisor` using it.

## Bi-encoder vs Cross-encoder

The retriever you have been using is a BI-ENCODER: it embeds the query once, embeds each chunk once, computes cosine similarity. Fast, but imperfect at deciding which of the top-10 is MOST relevant to THIS specific query.

A RERANKER is a CROSS-ENCODER: it looks at (query, chunk) as a pair and scores relevance directly. Much more accurate on the top few, at the cost of roughly 50-200 ms per candidate list.

The production pattern:

```mermaid
graph LR
    Q[Query] --> RET[Bi-encoder retrieve<br/>top 20]
    RET --> RR[Cohere Rerank 3.5<br/>Bedrock Rerank API]
    RR --> TOP3[top 3 by relevance]
    TOP3 --> LLM[Generator LLM]

    style RR fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000
```

Published 2026 guidance: reranking is the single highest-ROI addition to a basic RAG pipeline, typically 10-30% precision gain for 50-200 ms latency.

## Cohere Rerank 3.5 on Bedrock

The Bedrock Rerank API (`bedrock-agent-runtime.rerank()`) takes:
- A query string
- A list of text candidates (up to 1000 per call)
- A reranking model ARN

It returns the candidates re-ordered by relevance, with a `relevanceScore` per candidate. One API call covers the whole candidate list - not one call per chunk.

No marketplace subscription required. The `bedrock:Rerank` permission is what your
SageMaker execution role needs. The instructor verified this is enabled for the class
account in us-east-1 (Probe 3 in Cell 3 confirmed access).

In [ ]:
# =============================================================================
# DEMO: Cohere Rerank 3.5 via Bedrock Rerank API
# =============================================================================
# The Bedrock Rerank API takes a query + list of text sources and returns them
# re-ordered by relevance. It is NOT a generation call - it calls a dedicated
# cross-encoder model (Cohere Rerank 3.5) that scores each (query, chunk) pair.
#
# Why this is better than the bi-encoder retriever:
#   bi-encoder   : embed query once, embed each chunk once, cosine similarity.
#                  Fast, but imprecise for subtle relevance differences.
#   cross-encoder: score every (query, chunk) pair jointly.
#                  Slower per chunk but 10-30% more precise on the top results.
#
# Latency: ~100-200 ms per rerank call for up to 20 candidates.
# Cost: per-query pricing (one call covers all candidates in that call).

def bedrock_rerank(query_text: str, text_sources: list, num_results: int = 3) -> list:
    """Rerank text_sources by relevance to query_text using Cohere Rerank 3.5.

    Args:
        query_text:   The query string.
        text_sources: List of plain text strings (chunks) to rerank.
        num_results:  How many top results to return.

    Returns:
        List of dicts: [{index, relevance_score, text}, ...] sorted by score desc.
    """
    if not text_sources:
        return []

    response = bedrock_agent_runtime.rerank(
        queries=[
            {"type": "TEXT", "textQuery": {"text": query_text}}
        ],
        sources=[
            {
                "type": "INLINE",
                "inlineDocumentSource": {
                    "type": "TEXT",
                    "textDocument": {"text": s},
                },
            }
            for s in text_sources
        ],
        rerankingConfiguration={
            "type": "BEDROCK_RERANKING_MODEL",
            "bedrockRerankingConfiguration": {
                "numberOfResults": num_results,
                "modelConfiguration": {"modelArn": RERANK_MODEL_ARN},
            },
        },
    )

    out = []
    for r in response["results"]:
        out.append({
            "index":           r["index"],
            "relevance_score": r["relevanceScore"],
            "text":            text_sources[r["index"]],
        })
    return out


# Quick demo: rerank 5 local chunks from index_b by a test query
_demo_q = "What is the CTR reporting threshold for cash transactions?"
_candidates = [doc.page_content for doc, _ in
               index_b.similarity_search_with_score(_demo_q, k=5)]

_ranked = bedrock_rerank(_demo_q, _candidates, num_results=3)
print(f"Query: {_demo_q!r}")
print("Top-3 after Cohere Rerank 3.5:")
for r in _ranked:
    print(f"  score={r['relevance_score']:.3f}  text: {r['text'][:120].replace(chr(10),' ')}...")

In [ ]:
# =============================================================================
# DEMO: Measure the Rerank Lift on a Deliberately Ambiguous Query
# =============================================================================
# We pick a query where the bi-encoder's top hit is NOT the best chunk,
# then show Cohere Rerank 3.5 promoting the better chunk to rank 1.

q = "When must a bank report a large cash deposit to the government?"

# Bi-encoder baseline top-5
baseline = index_b.similarity_search_with_score(q, k=5)
print("BASELINE bi-encoder top-5 (index_b, 512/80 chunks):")
for i, (doc, score) in enumerate(baseline, 1):
    print(f"  #{i}  score={score:.3f}  source={doc.metadata['source']}")

# Rerank those 5 using Cohere Rerank 3.5
candidates = [doc.page_content for doc, _ in baseline]
ranked = bedrock_rerank(q, candidates, num_results=5)
print("\nAFTER Cohere Rerank 3.5 (same 5 candidates, reordered):")
sources = [baseline[r["index"]][0].metadata["source"] for r in ranked]
for i, r in enumerate(ranked, 1):
    print(f"  #{i}  rerank_score={r['relevance_score']:.3f}  source={sources[i-1]}")

print("\nObservation: the chunks do not change, but their order does.")
print("The reranker moved the most policy-relevant chunk to position 1.")

## Lab 2: Build policy_retriever_v2 with Cohere Reranking (15 min)

### Your Task

Improve the Week 17 `policy_retriever_agent` into `policy_retriever_v2` by
wrapping retrieval + Cohere reranking into a custom `@tool`, then building
an Agent that uses it.

### Steps

1. Write `reranked_retrieve(query, k_retrieve=10, k_final=3)`:
   - Call `bedrock_agent_runtime.retrieve()` directly with `STRANDS_KNOWLEDGE_BASE_ID`.
     (Do NOT use `strands_tools.retrieve` - it returns a pre-formatted string,
     not the raw `retrievalResults` list that `bedrock_rerank()` needs.)
   - Extract chunk texts from `retrievalResults[*].content.text`.
   - Extract source locations from `retrievalResults[*].location`.
   - Call `bedrock_rerank(query, candidates, num_results=k_final)`.
   - Attach `source_uri` to each ranked result and return the list.

2. Wrap it as a `@tool` called `retrieve_policy_reranked`:
   - The tool calls `reranked_retrieve` and formats the results as a string
     the agent can read (source URI + relevance score + text for each chunk).

3. Build `policy_retriever_v2 = Agent(model=llm, tools=[retrieve_policy_reranked], ...)`.
   Use the same system_prompt as `baseline_policy_retriever`.

4. Test: call `policy_retriever_v2("What is the CTR threshold?")` and print
   the first 400 characters. Compare to `baseline_policy_retriever` output.

### Expected Output

- `reranked_retrieve` returns a list of dicts with `relevance_score`, `text`, `source_uri`.
- `policy_retriever_v2` answers the question internally using Cohere reranking.
- The starter code already gives you the `@tool` wrapper and the comparison - you fill
  in `reranked_retrieve` and the `Agent(...)` call.

### Stretch (for fast finishers)

Time both agents with `time.perf_counter`. How much latency does reranking add
per query? At what query volume does that latency matter for a fraud ops team?

### Homework Extension

Replace `run_policy_retriever` in your Week 17 `lab3_supervisor` with a new
`run_policy_retriever_v2` tool that wraps `policy_retriever_v2`. Re-run the
three supervisor queries from Week 17 Lab 3. Does the supervisor's answer
quality change? Write one paragraph on whether the reranking lift shows up
in the full multi-agent loop.

In [ ]:
# =============================================================================
# SOLUTION: Lab 2 - Build policy_retriever_v2 with Cohere Reranking
# =============================================================================
# Key insight: we MUST call bedrock_agent_runtime.retrieve() directly, not
# strands_tools.retrieve. The Strands tool wraps its output in a ToolResult
# envelope and returns a pre-formatted string. bedrock_rerank() needs a plain
# list of text strings to score. The raw Bedrock API gives us retrievalResults
# which we can unpack into candidates + source location dicts.

def reranked_retrieve(query: str, k_retrieve: int = 10, k_final: int = 3) -> list:
    """Retrieve from the Week 17 KB then rerank with Cohere Rerank 3.5."""
    # Step 1: raw retrieval - get k_retrieve candidates from Bedrock KB
    raw = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": k_retrieve}},
    )
    items = raw.get("retrievalResults", [])

    # Step 2: extract plain text and source location from each result
    candidates = [i.get("content", {}).get("text", "") for i in items]
    sources    = [i.get("location", {}) for i in items]

    # Step 3: rerank - cross-encoder scores the (query, chunk) pairs
    ranked = bedrock_rerank(query, candidates, num_results=k_final)

    # Attach source URI to each ranked result so callers can cite it
    for r in ranked:
        r["source_uri"] = sources[r["index"]].get("s3Location", {}).get("uri", "unknown")
    return ranked


# Wrap as a @tool so the agent can call it
@tool
def retrieve_policy_reranked(query: str) -> str:
    """Retrieve fraud policy chunks from the KB, reranked by Cohere Rerank 3.5.

    Args:
        query: The policy question or investigation query.
    """
    ranked = reranked_retrieve(query, k_retrieve=10, k_final=3)
    lines = []
    for i, r in enumerate(ranked, 1):
        lines.append(f"[{i}] score={r['relevance_score']:.3f} | {r['source_uri']}")
        lines.append(r["text"])
        lines.append("")
    return "\n".join(lines)


# Build the improved agent (same prompt as Week 17, better tool)
policy_retriever_v2 = Agent(
    model=llm,
    tools=[retrieve_policy_reranked],
    system_prompt=(
        "You are a fraud policy specialist. Always call retrieve_policy_reranked "
        "before answering. Answer only from retrieved content. If the KB does not "
        "cover the question, say so and stop."
    ),
    callback_handler=None,
)

# Compare baseline vs v2 on one query
_test_q = "What is the CTR reporting threshold for cash transactions?"
print("--- baseline_policy_retriever ---")
print(str(baseline_policy_retriever(_test_q))[:400])
print("\n--- policy_retriever_v2 ---")
print(str(policy_retriever_v2(_test_q))[:400])

# Observation: policy_retriever_v2 sees the same KB content but the reranker
# promotes the most relevant CTR chunk to rank 1, giving the LLM a cleaner
# context to reason from.

> **Think About It #2**: Reranking adds 50-100 ms per query. In a fraud investigation workflow where the agent makes 3-5 retrieve calls per case, that is 150-500 ms of added latency per case. For an investigator reviewing 200 cases/day, is that worth the precision gain? What would change your answer: case volume, missed-policy cost, SLA contracts?

# Section 3: Evaluation - Step 3 of 3: Measuring the Improvement

## How Do We Know policy_retriever_v2 Is Actually Better?

Lab 2 gave you a v2 agent. This section gives you the measurement tool:
RAGAS. By the end of Lab 3 you will have RAGAS scores for both baseline
and v2, and you can make a data-backed claim about whether reranking helped.

## From "It Works on My Demo" to Numbers

Up to now our assessment of RAG quality has been "this looks right." That does not survive a compliance audit. RAGAS is an evaluation framework that scores RAG outputs with LLM-as-judge metrics.

## The Three Metrics We Will Use

| Metric | What it asks | Range | When it fails |
|--------|--------------|-------|---------------|
| `faithfulness` | Is the answer supported by the retrieved context? | 0-1 | Model hallucinated beyond what was retrieved |
| `answer_relevancy` | Does the answer address the user's question? | 0-1 | Model answered a different question |
| `context_precision` | Are retrieved chunks actually relevant? | 0-1 | Retriever returned off-topic chunks |

A fourth metric `context_recall` needs GROUND-TRUTH references for each question and is deferred to homework.

## LLM-as-Judge Architecture

```mermaid
graph LR
    Q[Question] --> R[Retriever]
    R --> C[Retrieved Context]
    C --> G[Generator LLM<br/>Claude Haiku]
    G --> A[Answer]

    Q --> J[RAGAS Judge LLM<br/>also Claude Haiku]
    C --> J
    A --> J
    J --> S[Scores:<br/>faithfulness<br/>answer_relevancy<br/>context_precision]

    style J fill:#fff3e0,stroke:#ff9800,stroke-width:2px,color:#000
```

We use Claude Haiku for both the generator AND the judge. This is a common cost-efficient pattern. More robust setups use a stronger judge model, which we discuss in the wrap-up.

## Bedrock + RAGAS Setup

RAGAS needs a judge LLM and an embedder. We wrap LangChain's Bedrock clients so RAGAS can call them:

```python
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm        = LangchainLLMWrapper(langchain_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(bedrock_embeddings)
```

Note on RAGAS v0.4: `LangchainLLMWrapper` is the stable documented path for Bedrock integration.

In [ ]:
# =============================================================================
# DEMO: Score One Retriever Answer with RAGAS
# =============================================================================
# RAGAS needs an EvaluationDataset. Each sample is a SingleTurnSample with
# user_input, response, and retrieved_contexts. We produce those three
# fields ourselves by calling bedrock-agent-runtime.retrieve() (for raw
# chunks) + Claude generator.

# Wrap the LangChain Bedrock clients for RAGAS
evaluator_llm        = LangchainLLMWrapper(langchain_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(bedrock_embeddings)

q = "What is the CTR reporting threshold?"

# Call Bedrock retrieve directly (Strands tool returns a pre-formatted string;
# we need the raw retrievalResults list for RAGAS context).
resp = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
    retrievalQuery={"text": q},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
)
retrieved_contexts = [it.get("content", {}).get("text", "")
                      for it in resp.get("retrievalResults", [])]

# Generate an answer (the same way the agent would, but direct)
gen_prompt = (
    "Answer using only the context.\n\n"
    f"Context:\n{chr(10).join(retrieved_contexts)}\n\nQuestion: {q}\nAnswer:"
)
answer = langchain_llm.invoke(gen_prompt).content

# Assemble a 1-sample EvaluationDataset
sample = SingleTurnSample(
    user_input         = q,
    response           = answer,
    retrieved_contexts = retrieved_contexts,
)
dataset = EvaluationDataset(samples=[sample])

# Evaluate
result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)
print("RAGAS result for single sample:")
print(result)

## Lab 3: RAGAS Score Sheet - Baseline vs policy_retriever_v2 (15 min)

### Your Task

Build an evaluation dataset of 5 fraud questions. Score both `baseline_policy_retriever`
and `policy_retriever_v2` with RAGAS on `faithfulness` and `answer_relevancy`.
Produce two DataFrames so you can see whether reranking helped.

### Steps

1. Define 5 fraud questions covering: CTR, OFAC, account takeover, structuring, high-risk MCCs.

2. Write `build_sample(question, use_v2=False)`:
   - If `use_v2=False`: call `bedrock_agent_runtime.retrieve()` for raw contexts,
     then call `langchain_llm.invoke()` to generate the answer.
   - If `use_v2=True`: call `reranked_retrieve(question, k_final=3)` for contexts
     (Cohere reranked), then call `langchain_llm.invoke()` for the answer.
   - Wrap into `SingleTurnSample(user_input, response, retrieved_contexts)`.
   - This lets you score BOTH configs with the same function.

3. Build `EvaluationDataset` for baseline (use_v2=False) and v2 (use_v2=True).

4. Evaluate each on `[faithfulness, answer_relevancy]`.

5. Convert results to DataFrames. Sort by faithfulness.

### Expected Output

Two DataFrames: `lab3_baseline_df` and `lab3_v2_df`.
If v2 faithfulness >= baseline faithfulness, reranking helped on this question set.

### Stretch (for fast finishers)

Write 3 adversarial questions - questions that sound reasonable but the KB does not cover.
Score them. What does the agent do when asked something the KB cannot answer? Does
`answer_relevancy` catch it?

### Homework Extension

Write `gate(df, min_faith=0.8, min_rel=0.7) -> bool` that returns True only if the
MINIMUM score across all rows passes BOTH thresholds. Wrap it around the evaluation.
Which metric threshold do you trust more for a compliance use case, and why?

In [ ]:
# =============================================================================
# SOLUTION: Lab 3 - RAGAS Score Sheet: Baseline vs policy_retriever_v2
# =============================================================================
# build_sample calls bedrock_agent_runtime.retrieve() directly because
# RAGAS needs a list of plain strings for retrieved_contexts, not the
# formatted ToolResult string that strands_tools.retrieve returns.
# The use_v2=True path swaps in reranked_retrieve (Cohere Rerank 3.5).

def build_sample(question: str, use_v2: bool = False) -> SingleTurnSample:
    """Retrieve + generate an answer + wrap into a SingleTurnSample for RAGAS."""
    if use_v2:
        # Cohere Rerank 3.5 path: reranked_retrieve returns list of dicts
        ranked   = reranked_retrieve(question, k_final=3)
        contexts = [r["text"] for r in ranked]
    else:
        # Baseline path: raw Bedrock retrieve
        resp = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=STRANDS_KNOWLEDGE_BASE_ID,
            retrievalQuery={"text": question},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 3}},
        )
        contexts = [i.get("content", {}).get("text", "")
                    for i in resp.get("retrievalResults", [])]

    # Generate an answer grounded in the retrieved context
    prompt = (
        "Answer using only the context.\n\n"
        f"Context:\n{chr(10).join(contexts)}\n\n"
        f"Question: {question}\nAnswer:"
    )
    answer = langchain_llm.invoke(prompt).content

    return SingleTurnSample(
        user_input=question,
        response=answer,
        retrieved_contexts=contexts,
    )


lab3_questions = [
    "What is the CTR reporting threshold for cash transactions?",
    "When must a bank screen wire transfers against the OFAC SDN list?",
    "What indicates account takeover following a password change?",
    "What cash deposit pattern constitutes structuring?",
    "Which merchant category codes are classified as high risk?",
]

# Build samples for BOTH configs
baseline_samples = [build_sample(q, use_v2=False) for q in lab3_questions]
v2_samples       = [build_sample(q, use_v2=True)  for q in lab3_questions]

# Evaluate each config
baseline_result = evaluate(
    EvaluationDataset(samples=baseline_samples),
    metrics=[faithfulness, answer_relevancy],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)
v2_result = evaluate(
    EvaluationDataset(samples=v2_samples),
    metrics=[faithfulness, answer_relevancy],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

# Convert to DataFrames and sort by faithfulness to see worst-case rows first
lab3_baseline_df = (baseline_result.to_pandas()
                    [["user_input", "faithfulness", "answer_relevancy"]]
                    .sort_values("faithfulness"))
lab3_v2_df       = (v2_result.to_pandas()
                    [["user_input", "faithfulness", "answer_relevancy"]]
                    .sort_values("faithfulness"))

print("Baseline scores:")
print(lab3_baseline_df.to_string(index=False))
print("\npolicy_retriever_v2 scores (Cohere Rerank 3.5):")
print(lab3_v2_df.to_string(index=False))

> **Think About It #3**: Claude Haiku is both the generator AND the judge in your Lab 3. That is cheap and common, but it is also a form of self-evaluation. Name two failure modes this could hide. When would you pay for Claude Sonnet (or another model) as the judge, and how would you budget for it?

# Section 4: A/B Comparison + week18_supervisor (MAIN OUTCOME)

## Closing the Week 17-18 Arc

You have:
- `policy_retriever_v2` (reranked KB retrieval, built in Lab 2)
- `baseline_case_agent` (same as Week 17, re-declared in Cell 4)
- RAGAS scores comparing baseline vs v2 (from Lab 3)

Lab 4 has two parts:
- **Part A** - declare the winner with numbers using the two Lab 3 DataFrames
- **Part B** - build `week18_supervisor` using `policy_retriever_v2` + `mem0_memory`,
  the upgraded version of Week 17's `lab3_supervisor`

## The Comparison Matrix

```mermaid
graph TB
    subgraph "Config 1: Baseline"
        B1[Bedrock KB<br/>default bi-encoder retrieval<br/>no rerank]
    end

    subgraph "Config 2: policy_retriever_v2"
        B2[Bedrock KB<br/>bi-encoder retrieve<br/>+ Cohere Rerank 3.5]
    end

    B1 --> EVAL[RAGAS<br/>faithfulness<br/>answer_relevancy]
    B2 --> EVAL
    EVAL --> DF[pandas DataFrame<br/>config x metric<br/>pick winner]
    DF --> SUP[week18_supervisor<br/>using winning retriever]

    style DF fill:#e8f5e9,stroke:#2e7d32,stroke-width:3px,color:#000
    style SUP fill:#e8f5e9,stroke:#2e7d32,stroke-width:3px,color:#000
```

## What We Deliberately Keep Simple This Week

- Only 2 configs (baseline vs reranked). Adding more is the stretch.
- Only 3 questions. More is the stretch.
- No experiment tracking tool (MLflow/DVC) - Week 19's job.
- No A/B test with live traffic - Week 20's job (Langfuse observability).

In [ ]:
# =============================================================================
# DEMO: Evaluate Baseline vs Reranked Side by Side
# =============================================================================
# Reuse the `build_sample` function from Lab 3 as the baseline generator.
# Create a `build_sample_reranked` variant that uses `reranked_retrieve`.
# Score each on 3 questions, produce a single comparison DataFrame.

def build_sample_reranked(question):
    ranked = reranked_retrieve(question, k_retrieve=10, k_final=3)
    contexts = [r["text"] for r in ranked]
    prompt = (
        "Answer using only the context.\n\n"
        f"Context:\n{chr(10).join(contexts)}\n\n"
        f"Question: {question}\nAnswer:"
    )
    answer = langchain_llm.invoke(prompt).content
    return SingleTurnSample(
        user_input=question,
        response=answer,
        retrieved_contexts=contexts,
    )


demo_questions = [
    "What is the CTR reporting threshold?",
    "When must a bank file a CTR on aggregated transactions?",
    "What evidence indicates account takeover?",
]


def score_config(config_name, sample_builder, questions):
    """Score one RAG config across a list of questions. Returns a DataFrame."""
    ds = EvaluationDataset(samples=[sample_builder(q) for q in questions])
    r = evaluate(
        dataset=ds,
        metrics=[faithfulness, answer_relevancy],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )
    df = r.to_pandas()
    df["config"] = config_name
    return df[["config", "user_input", "faithfulness", "answer_relevancy"]]


base_scores   = score_config("baseline", build_sample,           demo_questions)
rerank_scores = score_config("+ rerank", build_sample_reranked,  demo_questions)
compare = pd.concat([base_scores, rerank_scores], ignore_index=True)
print(compare.to_string(index=False))

print("\nMean scores per config:")
print(compare.groupby("config")[["faithfulness", "answer_relevancy"]]
               .mean().round(3))

## Main Lab 4: A/B Winner + week18_supervisor (15 min)

### Part A: Declare the Winner (5 min)

1. Use `lab3_baseline_df` and `lab3_v2_df` from Lab 3 (or the safety-net).
2. Build `lab4_comparison_df` with columns: `config`, `faithfulness`, `answer_relevancy`
   showing MEAN scores per config.
3. Print the winner - the config with the higher MEAN across both metrics.
4. In a MARKDOWN CELL below your code cell, write 3-4 sentences explaining which
   config won, by how much, and why you would (or would not) ship it.

### Part B: Build week18_supervisor (10 min)

5. Wrap `policy_retriever_v2` as a `@tool` called `run_policy_retriever_v2`:
   - The tool calls `policy_retriever_v2(question)` and returns the string.
6. Build `week18_supervisor = Agent(...)` with two tools:
   - `run_policy_retriever_v2` (your improved retriever)
   - `mem0_memory` (same as Week 17's supervisor)
   Use the same system_prompt as Week 17's `lab3_supervisor`.
7. Test with one multi-step query to confirm it works end-to-end.

### Expected Output

- `lab4_comparison_df` with winner printed.
- `week18_supervisor` answering a fraud investigation query with policy citations.

### Stretch (for fast finishers)

Add a `latency_ms` column by timing each sample's end-to-end generation.
Does the "best" RAGAS config change when you factor latency in?

### Homework Extension

Run the same 3 queries you used in Week 17 Lab 3 through `week18_supervisor`.
Compare the answers side-by-side. Write one paragraph: did the retrieval improvement
show up in the final supervisor response, or was the baseline already "good enough"?

In [ ]:
# =============================================================================
# SOLUTION: Main Lab 4 - A/B Winner + week18_supervisor
# =============================================================================

# ---------------------------------------------------------------------------
# PART A: Declare the winner using Lab 3 DataFrames
# ---------------------------------------------------------------------------
# Add a "config" column to each DataFrame, concat, compute means, pick winner.

_base_with_config = lab3_baseline_df.copy()
_base_with_config["config"] = "baseline"

_v2_with_config = lab3_v2_df.copy()
_v2_with_config["config"] = "policy_retriever_v2"

lab4_comparison_df = pd.concat([_base_with_config, _v2_with_config], ignore_index=True)
means = (lab4_comparison_df
         .groupby("config")[["faithfulness", "answer_relevancy"]]
         .mean()
         .round(3))
means["combined"] = means.mean(axis=1).round(3)
winner = means["combined"].idxmax()

print("Mean scores per config:")
print(means)

faith_delta = (means.loc["policy_retriever_v2", "faithfulness"]
               - means.loc["baseline", "faithfulness"])
print(f"\nWinner: {winner} (faithfulness {faith_delta:+.3f} vs baseline)")

# ---------------------------------------------------------------------------
# PART B: Build week18_supervisor
# ---------------------------------------------------------------------------

@tool
def run_policy_retriever_v2(question: str) -> str:
    """Ask the improved fraud policy agent (Cohere-reranked KB retrieval).

    Args:
        question: A natural-language fraud policy question.
    """
    return str(policy_retriever_v2(question))


week18_supervisor = Agent(
    model=llm,
    tools=[run_policy_retriever_v2, mem0_memory],
    system_prompt=(
        "You are a senior fraud investigator. For every question: "
        "1) Search your memory for prior findings on this case or customer. "
        "2) Retrieve the relevant fraud policies using run_policy_retriever_v2. "
        "3) Synthesize a clear compliance recommendation with policy citations. "
        "Always save your key findings to memory after answering."
    ),
    callback_handler=None,
)

print("\nweek18_supervisor ready.")
print("Tools: run_policy_retriever_v2 (Cohere Rerank 3.5), mem0_memory")

# Test with one multi-step query
_sv_q = ("Customer made 3 cash deposits of $9,400 each on consecutive days. "
         "Is this structuring? What policy applies?")
print("\nweek18_supervisor response:")
print(str(week18_supervisor(_sv_q))[:600])

> **Think About It #4**: You have numbers now. The baseline vs reranked comparison is probably a small lift (a few tenths on each metric). In a real code review, a reviewer will ask: "Why should we pay the 50-100 ms per query and the per-query Cohere fee for that lift?" What is the minimum lift you would demand before shipping a RAG change to production at a financial institution, and what non-numeric evidence (compliance, auditability, risk) would you bring to the review?

## Running the Fraud Agent UI

`week18_supervisor` above is the backend for a standalone Gradio chat app.

Run it from the SageMaker terminal:

```bash
python fraud_agent_ui.py
```

Or from a notebook cell:

```python
!python fraud_agent_ui.py &
```

Gradio will print a URL like `http://0.0.0.0:7860`. In SageMaker Studio, access
it through the Studio proxy URL that Gradio prints at startup.

`fraud_agent_ui.py` is in the same folder as this notebook. It is self-contained -
no imports from this notebook. You can carry it directly into Weeks 19-20 for
DVC versioning and CI/CD integration.

---

**The arc is complete:**
- Week 17: built the pipeline (agentic RAG + case memory)
- Week 18: measured and improved it (chunking + reranking + RAGAS + full supervisor)
- Week 19: version and track it (DVC + MLflow)
- Week 20: observe it in production (Langfuse)

The `fraud_agent_ui.py` is what you are versioning next week.

## Optional Notebooks (in this folder)

- `week_18_optional_hybrid_search.ipynb` - BM25 + FAISS + Reciprocal Rank Fusion.
  No new installs needed (`rank_bm25` is already installed).
- `week_18_optional_deep_eval.ipynb` - DeepEval 6-metric evaluation suite with
  a custom Claude Haiku 3 judge (bypasses the native BedrockModel class which
  targets Sonnet-class model IDs not available in the class account).

# Summary: What We Learned Today

## Key Takeaways

### Chunking
- Recursive 512/80 is the benchmark default; 1024/150 wins on cross-section fraud/finance queries. Measure, do not assume.
- Bedrock Knowledge Bases accept FIXED / HIERARCHICAL / SEMANTIC / NONE as `ChunkingConfiguration`. Hierarchical + hybrid search + reranking is the AWS-recommended production default.

### Reranking
- Cohere Rerank 3.5 is a cross-encoder model accessed via `bedrock-agent-runtime.rerank()` - no marketplace subscription needed.
- One API call covers the full candidate list (not one call per chunk). Latency: 100-200 ms for up to 20 candidates.
- Published 2026 data: 10-30% precision gain over bi-encoder alone for 100-200 ms latency cost.

### RAGAS v0.4
- Faithfulness, answer_relevancy, context_precision as the main three.
- `LangchainLLMWrapper(ChatBedrockConverse(...))` as the judge - no OpenAI key needed.
- `context_recall` needs ground-truth references - homework.

### Progressive Agent Improvement Arc

- Week 17: built `policy_retriever_agent` (raw Bedrock KB, default chunking, no reranking)
- Section 1: measured how chunk size affects what the agent retrieves (local FAISS + real Bedrock KBs after break)
- Section 2: replaced the bi-encoder ordering with Cohere Rerank 3.5 via Bedrock Rerank API
- Lab 2: built `policy_retriever_v2` - same Agent interface, better `@tool` under the hood
- Lab 3: proved the improvement with RAGAS (faithfulness + answer_relevancy, side-by-side)
- Lab 4 Part A: picked the winner with data, wrote the justification
- Lab 4 Part B: wired `policy_retriever_v2` into `week18_supervisor` - the upgraded version
  of Week 17's `lab3_supervisor`, ready for Week 19 MLOps versioning

### week18_supervisor
- Two tools: `run_policy_retriever_v2` (Cohere-reranked KB) + `mem0_memory` (per-investigator case history).
- Same architecture as Week 17's supervisor, now with measured retrieval quality under it.
- Run `fraud_agent_ui.py` from the SageMaker terminal to chat with it in a browser.

## Looking Ahead

- **Week 19 (MLOps Part 1)** introduces DVC for data versioning and MLflow for experiment tracking. Every comparison DataFrame you built today would live in MLflow next week, and every eval dataset would be DVC-versioned.
- **Week 20 (MLOps Part 2)** adds online observability with Langfuse around the exact same supervisor. Today's RAGAS scores are OFFLINE; Week 20's are ONLINE - on real production traffic.
- **Weeks 21-22 (Airflow)** schedule periodic re-evaluation runs so your RAGAS numbers stay fresh as the KB grows.
- **Week 23 (Ethics)** connects today's `faithfulness` metric to GDPR and EU AI Act audit requirements - citations from retrieve + RAGAS faithfulness scores are exactly what an auditor will ask for.
- **Week 24 (Capstone)** - an evaluated, reranked, retrieval-augmented multi-agent system is one of the three canonical capstone paths.

## Homework

### Homework 1: Bedrock HIERARCHICAL Chunking (Lab 1 extension)
Create a second Bedrock KB with `ChunkingConfiguration=HIERARCHICAL` (parent 1500, child 300, overlap 60). Ingest the same 8 policy docs. Compare retrieval quality vs your local Lab 1 results. Why might Bedrock's hierarchical retrieval score differently than local recursive splitting?

### Homework 2: Latency-Aware Rerank (Lab 2 extension)
Time your `reranked_retrieve` with `time.perf_counter`. Report latency overhead per query. For a fraud supervisor that makes 3-5 retrieve calls per case and serves 200 cases/day, what is the daily wall-clock cost of reranking?

### Homework 3: CI Gate + context_recall (Lab 3 extension)
Implement `gate(df, min_faith=0.8, min_rel=0.7) -> bool`. Also add `context_recall` as a fourth metric - this requires authoring ground-truth references per question. Which of the four metrics do you put on the CI gate, and which do you track but not block on?

### Homework 4: Roll Winner into Week 17 Supervisor (Lab 4 extension - CRITICAL)
Take the winner of your Lab 4 comparison. Replace `run_policy_retriever` in Week 17's `lab3_supervisor` with `run_policy_retriever_v2`. Re-run Week 17's Lab 3 multi-retriever supervisor end-to-end. Does the supervisor's final decision quality change? Write one paragraph on whether the offline lift (what we measured this week) survives the agent loop.

## Great Work Today

**The journey so far:**
- Weeks 11-14: Make LLMs respond (prompt, fine-tune, classify)
- Weeks 15-16: Make LLMs ACT (tools, memory, supervisor multi-agent)
- Week 17: Make LLMs LOOK THINGS UP (agentic RAG with Bedrock KB + FAISS)
- **Week 18: Measure and improve what they look up (chunking + Cohere reranking + RAGAS + upgraded supervisor)**

**Next up**: Week 19 - MLOps for the RAG pipeline you just tuned.